In [27]:
from dotenv import load_dotenv

# Don't need to specify full path if notbook file and .env file are on the same leve in the project structure
load_dotenv("../.env")

True

In [57]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_aws import BedrockEmbeddings, ChatBedrockConverse
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [3]:
video_url = "https://www.youtube.com/watch?v=6Fa91SY9Gnw"

In [5]:
youtube_vid_loader = YoutubeLoader.from_youtube_url(video_url)
vid_dcos = youtube_vid_loader.load()

In [13]:
transcript = vid_dcos[0].page_content
len(transcript)

6035

In [14]:
transcript_stripped = transcript.replace(u'\xa0', u'').replace(u'\uf0a7', u'')
transcript_stripped

'According to the TIOBE Index for January 2022, Python is the programming language with the\nhighest increase in ratings in one year, making it officially the top choice\nof employers and professionals alike. In fact, it has become a key prerequisite\nto landing a job in data science. Luckily, Python is also very beginner friendly. So, if\nyou’re an aspiring data analyst or data scientist, learning Python is certainly the way to go.\nThat said, our job is to make every step of your learning curve easy and enjoyable. So, If you’re\nstill new to Python, you might want to start for free with our Introduction to Python course.\nWe’ve provided a link in the description. But if you already have some basic Python experience and\nwant to take it to the next level, in this video you’ll discover 4 interesting Python projects\nfor beginners – all including code for practice! Great! So, let’s start with project\nnumber 1 - Uber Trips Analysis Since it was founded in 2009, Uber has become one\nof t

In [15]:
len(transcript_stripped)

5853

In [18]:
char_text_spliter = CharacterTextSplitter(separator=".", chunk_size=500, chunk_overlap=0)

In [23]:
transcript_split = char_text_spliter.split_text(transcript_stripped)
transcript_split

Created a chunk of size 580, which is longer than the specified 500


['According to the TIOBE Index for January 2022, Python is the programming language with the\nhighest increase in ratings in one year, making it officially the top choice\nof employers and professionals alike. In fact, it has become a key prerequisite\nto landing a job in data science. Luckily, Python is also very beginner friendly. So, if\nyou’re an aspiring data analyst or data scientist, learning Python is certainly the way to go',
 'That said, our job is to make every step of your learning curve easy and enjoyable. So, If you’re\nstill new to Python, you might want to start for free with our Introduction to Python course.\nWe’ve provided a link in the description',
 'But if you already have some basic Python experience and\nwant to take it to the next level, in this video you’ll discover 4 interesting Python projects\nfor beginners – all including code for practice! Great! So, let’s start with project\nnumber 1 - Uber Trips Analysis Since it was founded in 2009, Uber has become one

In [24]:
len(transcript_split)

15

In [30]:
embedding = BedrockEmbeddings(model_id="amazon.titan-embed-text-v1", region_name="us-east-1")

In [34]:
vectorstore = Chroma.from_texts(persist_directory="./python-projects",
                                embedding=embedding,
                                texts=transcript_split)

In [36]:
vectorstore.get()['documents']

['According to the TIOBE Index for January 2022, Python is the programming language with the\nhighest increase in ratings in one year, making it officially the top choice\nof employers and professionals alike. In fact, it has become a key prerequisite\nto landing a job in data science. Luckily, Python is also very beginner friendly. So, if\nyou’re an aspiring data analyst or data scientist, learning Python is certainly the way to go',
 'That said, our job is to make every step of your learning curve easy and enjoyable. So, If you’re\nstill new to Python, you might want to start for free with our Introduction to Python course.\nWe’ve provided a link in the description',
 'But if you already have some basic Python experience and\nwant to take it to the next level, in this video you’ll discover 4 interesting Python projects\nfor beginners – all including code for practice! Great! So, let’s start with project\nnumber 1 - Uber Trips Analysis Since it was founded in 2009, Uber has become one

In [38]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.3},
)

In [39]:
question = "Could you tell me all essential beginner Python projects according to the video?"

In [40]:
retrieved_docs = retriever.invoke(question)

In [43]:
for i in retrieved_docs:
    print(f"Page Content: {i.page_content}\n----------\n")

Page Content: But if you already have some basic Python experience and
want to take it to the next level, in this video you’ll discover 4 interesting Python projects
for beginners – all including code for practice! Great! So, let’s start with project
number 1 - Uber Trips Analysis Since it was founded in 2009, Uber has become one
of the most famous unicorn companies, offering its services to more than 80 countries worldwide
----------

Page Content: Data science has countless applications in healthcare, and one of them is to
predict certain diseases, especially chronic ones. Consider this. If you’re planning to go out, you
normally check the weather news the day before so that you can prepare. if it’s going to rain,
you’ll grab an umbrella and wear a raincoat. And if the forecast says it is going to be hot,
you’ll wear light clothes. The same approach is valid for disease prevention
----------

Page Content: This
unreasonable behavior caused many deaths, especially in hospitals where m

In [48]:
TEMPLATE_1 = """You are a helpful chatbot that answers questions on YouTube videos.
Answer the questions using only the following context:
{context}"""

TEMPLATE_2 = """{question}"""

In [49]:
message_template_1 = SystemMessagePromptTemplate.from_template(TEMPLATE_1)
message_template_2 = HumanMessagePromptTemplate.from_template(TEMPLATE_2)

In [51]:
chat_template = ChatPromptTemplate.from_messages([
    message_template_1,
    message_template_2
])

In [62]:
chat = ChatBedrockConverse(
    model="us.anthropic.claude-sonnet-4-6",
    region_name="us-east-1",
    max_tokens=250,
    temperature=0
)

In [63]:
str_parser = StrOutputParser()

In [64]:
chain = {
    "context": retriever,
    "question": RunnablePassthrough(),
} | chat_template | chat | str_parser

In [65]:
response = chain.invoke(question)

In [67]:
print(response)

Based on the provided context, only **one** beginner Python project is explicitly mentioned:

1. **Uber Trips Analysis** - A project analyzing Uber trip data, noting that Uber was founded in 2009 and now operates in more than 80 countries worldwide.

The context mentions that there are **4 interesting Python projects for beginners** in the video, but only the first one (Uber Trips Analysis) is detailed in the available text. The other three projects are not described in the provided context, so I'm unable to share details about them.
